# ARIMA Order Selection

Wiki reference for [ARIMA order selection](https://ml-viz-ruby.vercel.app/wiki/arima-order-selection).

**The idea in one sentence.** Choosing $(p, d, q)$ for an ARIMA model is a recipe: **difference**
until the ADF test says stationary (that fixes $d$), read the **ACF/PACF** to propose $p$ and
$q$, then pick among candidates by **AIC/BIC** and confirm the residuals are **white noise**
(Ljung-Box).

We walk the full recipe on a trending sales series, **validate the differencing order, the
model choice, and residual whiteness**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings; warnings.filterwarnings('ignore')
plt.style.use('dark_background')


## The 36-observation worked dataset

Drawn from an actual ARIMA(0,1,1)-with-drift process ($\theta_1 = -0.6$, drift $2.0$,
$\sigma = 3$) and rounded to integers, so the procedure below has something real to recover.
That matters more than it sounds: a series that merely *looks* trending — a deterministic ramp,
say — has no unit root to test and no MA structure to find, and every diagnostic below would
report nonsense while appearing to work.


In [ ]:
y = np.array([
    50,51,54,55,60,56,64,63,65,64,63,67,
    72,73,77,77,80,81,80,84,85,90,90,92,
    96,99,99,101,104,109,106,106,119,115,118,114
], dtype=float)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(y, color='#14b8a6')
ax.set_title('Monthly sales — 36 observations (upward trend visible)')
ax.set_xlabel('Month'); ax.set_ylabel('Sales')
plt.tight_layout(); plt.show()


## Phase 1a: ADF test to determine d

In [ ]:
stat_raw, p_raw = adfuller(y)[:2]
stat_d, p_d = adfuller(np.diff(y))[:2]
print(f'Raw series:   ADF={stat_raw:.2f}  p={p_raw:.4f}')
print(f'  -> {"non-stationary (d += 1)" if p_raw > 0.05 else "stationary"}')
print(f'First diff:   ADF={stat_d:.2f}  p={p_d:.4f}')
print(f'  -> {"non-stationary" if p_d > 0.05 else "stationary ✓  d=1"}')


### Validate: differencing once makes the series stationary (d = 1)

The ADF null hypothesis is "has a unit root" (non-stationary). The raw trending series fails to
reject (p > 0.05), but the first difference rejects strongly (p < 0.05) — so $d = 1$. We confirm.

In [ ]:
print(f'raw series ADF p={p_raw:.4f}; first difference ADF p={p_d:.4f}')
assert p_raw > 0.05, 'the raw series is non-stationary (trend) — ADF fails to reject a unit root'
assert p_d < 0.05, 'the first difference is stationary — ADF rejects the unit root -> d=1'
print('\n✅ difference until stationary: here one difference suffices, so d=1')

## Phase 1b: ACF/PACF of differenced series

In [ ]:
dy = np.diff(y)
acf_v = acf(dy, nlags=12)
pacf_v = pacf(dy, nlags=12)
sig = 1.96 / np.sqrt(len(dy))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
lags = np.arange(len(acf_v))
ax1.bar(lags, acf_v, color='steelblue', alpha=0.8)
ax1.axhline(sig, color='yellow', ls='--'); ax1.axhline(-sig, color='yellow', ls='--')
ax1.set_title('ACF of Δy — cuts off at lag 1 → suggests MA(1)')
ax2.bar(lags, pacf_v, color='#6366f1', alpha=0.8)
ax2.axhline(sig, color='yellow', ls='--'); ax2.axhline(-sig, color='yellow', ls='--')
ax2.set_title('PACF of Δy — tails off → confirms MA order, not AR')
plt.tight_layout(); plt.show()


## Phase 2+3: Fit candidates, compare AIC/BIC

In [ ]:
header = f"{'Model':<20} {'AIC':>8} {'BIC':>8} {'LjungBox-p':>12}"
print(header)
print('-' * len(header))
for order in [(0,1,1), (1,1,0), (1,1,1)]:
    res = ARIMA(y, order=order, trend='t').fit()
    lb_p = acorr_ljungbox(res.resid, lags=10, return_df=True)['lb_pvalue'].min()
    ok = 'OK' if lb_p > 0.05 else 'FAIL'
    label = f'ARIMA{str(order)}'
    print(f'{label:<20} {res.aic:>8.1f} {res.bic:>8.1f} {lb_p:>11.3f}  {ok}')


### Validate: AIC picks the model the ACF/PACF suggested

The ACF of $\Delta y$ cut off at lag 1 (→ MA(1)) and the PACF tailed off (not AR), pointing to
**ARIMA(0,1,1)**. Information criteria should agree, and the chosen model's residuals should be
white noise (Ljung-Box p > 0.05). We confirm both.

In [ ]:
cand = {o: ARIMA(y, order=o, trend='t').fit() for o in [(0,1,1), (1,1,0), (1,1,1)]}
aics = {o: r.aic for o, r in cand.items()}
best_o = min(aics, key=aics.get)
print('AIC by model:', {str(o): round(a,1) for o,a in aics.items()})
assert best_o == (0,1,1), 'AIC agrees with the ACF/PACF diagnosis: ARIMA(0,1,1)'
lb = acorr_ljungbox(cand[(0,1,1)].resid, lags=10, return_df=True)['lb_pvalue'].min()
assert lb > 0.05, 'ARIMA(0,1,1) residuals are indistinguishable from white noise (Ljung-Box)'
print(f'\n✅ ARIMA(0,1,1): lowest AIC and white-noise residuals (Ljung-Box p={lb:.2f})')

## Phase 4: 6-step forecast with prediction intervals

In [ ]:
best = ARIMA(y, order=(0,1,1), trend='t').fit()
fc = best.get_forecast(steps=6)
fc_df = fc.summary_frame(alpha=0.05)
print(fc_df[['mean','mean_ci_lower','mean_ci_upper']].round(1))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(36), y, color='#14b8a6', label='Observed')
ax.plot(range(36, 42), fc_df['mean'], 'o--', color='#6366f1', label='Forecast')
ax.fill_between(range(36, 42),
                fc_df['mean_ci_lower'], fc_df['mean_ci_upper'],
                alpha=0.3, color='#6366f1', label='95% PI')
ax.axvline(35.5, color='gray', ls=':')
ax.legend(); ax.set_title('ARIMA(0,1,1) — 6-step forecast')
plt.tight_layout(); plt.show()


### The trap: `d >= 1` fits no constant by default

Every fit above passes `trend='t'`. Without it, statsmodels fits **no drift term** when
$d \ge 1$ — and on a trending series that produces a flat forecast sitting below the last
observation, with a much worse AIC. It does not warn you.


In [ ]:
no_drift = ARIMA(y, order=(0,1,1)).fit()            # note: no trend='t'
with_drift = ARIMA(y, order=(0,1,1), trend='t').fit()

print(f'no drift  : AIC={no_drift.aic:6.1f}  forecast={np.round(no_drift.get_forecast(3).predicted_mean, 1)}')
print(f'with drift: AIC={with_drift.aic:6.1f}  forecast={np.round(with_drift.get_forecast(3).predicted_mean, 1)}')
print(f'\nlast observation: {y[-1]:.0f}')

assert no_drift.aic > with_drift.aic, 'dropping the drift term costs real likelihood'
assert np.allclose(no_drift.get_forecast(3).predicted_mean.std(), 0, atol=1e-8), \
    'without a drift term the forecast is flat forever'
print('\n\u2705 if your ARIMA forecast is suspiciously flat, check the trend argument first')


## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **over-differencing** | inflates variance, adds spurious MA (demo) — stop when ADF passes |
| **AIC vs BIC** | BIC penalises parameters harder → prefers simpler models |
| **ignoring residual checks** | a good AIC with autocorrelated residuals is misspecified |
| **short series** | few points → unreliable ACF/PACF and unstable fits |
| **seasonality** | pure ARIMA misses seasonal structure; use SARIMA |

Demo: a second difference inflates variance — the over-differencing signature.

In [ ]:
# The most common ARIMA mistake is OVER-differencing. Differencing a series that is already
# stationary inflates its variance and injects a spurious negative MA(1) term — a sign you went
# one difference too far. Here d=1 is correct; a second difference makes the variance jump.
v1 = np.var(np.diff(y))
v2 = np.var(np.diff(np.diff(y)))
print(f'variance of 1st difference: {v1:.1f}')
print(f'variance of 2nd difference: {v2:.1f}  (inflated -> over-differenced)')
assert v2 > v1, 'a second difference inflates variance -> you have over-differenced'
print('\nStop differencing as soon as ADF says stationary — over-differencing hurts the fit.')

## ✏️ Your turn

Add ARIMA(2,1,0) to the comparison. Does it improve on ARIMA(0,1,1) by AIC?


In [ ]:
# TODO(you): fit ARIMA(2,1,0) and print AIC, BIC, Ljung-Box p-value
res_210 = None  # replace with fitted model

assert res_210 is not None, 'Fit the model first!'
print(f'ARIMA(2,1,0) AIC = {res_210.aic:.1f}')


<details>
<summary>Solution</summary>

```python
res_210 = ARIMA(y, order=(2,1,0), trend='t').fit()
lb_p = acorr_ljungbox(res_210.resid, lags=10, return_df=True)['lb_pvalue'].min()
print(f'ARIMA(2,1,0): AIC={res_210.aic:.1f}  BIC={res_210.bic:.1f}  p={lb_p:.3f}')
```

</details>


## Key takeaways

- **Fix $d$ by differencing** until ADF says stationary — here $d = 1$ (verified).
- **Read $p, q$ from ACF/PACF:** ACF cut-off → MA order, PACF cut-off → AR order.
- **Choose by AIC/BIC** and confirm **white-noise residuals** (Ljung-Box) — ARIMA(0,1,1) wins
  here (verified).
- **Don't over-difference:** a needless extra difference inflates variance (demo).